# SGLang

A practical reference for **SGLang** (Structured Generation Language) — a fast serving framework for large language and vision-language models. SGLang pairs a high-performance runtime (**SRT**, the SGLang Runtime) with a Python-embedded **frontend DSL** for expressing complex, multi-step generation programs.

> Repository: <https://github.com/sgl-project/sglang> · Docs: <https://docs.sglang.ai>

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

SGLang is an open-source framework for **serving and programming LLMs and vision-language models (VLMs)** at high throughput. It originated from research at UC Berkeley (the team behind vLLM, Chatbot Arena, and LMSYS) and has become one of the fastest production serving stacks, used to serve frontier open models (Llama, Qwen, DeepSeek, Mixtral, Gemma, and VLMs such as LLaVA and Qwen-VL).

### What is it?

SGLang has two tightly integrated halves:

- **SRT — the SGLang Runtime**: a high-performance inference engine and OpenAI-compatible server. Its signature optimization is **RadixAttention**, which automatically reuses the KV cache of shared prompt prefixes across requests via a radix tree.
- **The frontend language**: a Python-embedded DSL (`sgl.function`, `sgl.gen`, `sgl.select`, `fork`/`join`) for writing multi-step generation programs — branching, parallelism, control flow, and constrained output — that the runtime co-optimizes with the cache.

### Why use it?

- **Automatic prefix caching (RadixAttention).** Shared prefixes (system prompts, few-shot exemplars, multi-turn history, tree-of-thought branches) are cached and reused with no manual bookkeeping — often a multi-x speedup on prefix-heavy workloads.
- **Fast structured/constrained decoding.** Built-in JSON-schema, regex, and grammar (EBNF) constraints via a compressed finite-state machine and the `xgrammar` backend, with little throughput penalty.
- **High raw throughput.** Continuous batching, paged KV cache, CUDA graphs, FlashInfer/Triton attention kernels, and chunked prefill put SRT among the top open serving stacks.
- **A real programming model.** The frontend DSL makes agentic / multi-call patterns (parallel branches, self-consistency, tool loops) concise instead of hand-rolled.

### When to use it?

- Prompt-heavy or multi-turn workloads where many requests **share a large common prefix**.
- You need **fast JSON / regex / grammar-constrained output** at serving scale.
- You're building **agentic or multi-step generation programs** and want a runtime that co-optimizes them.
- You want a drop-in **OpenAI-compatible** endpoint with top-tier throughput.

## Key Features

### Core Capabilities of SGLang

| Feature | Description | Benefit |
|---------|-------------|---------|
| RadixAttention | Automatic KV-cache reuse of shared prefixes via a radix tree over cached tokens | Large speedups on shared system prompts, few-shot, multi-turn, tree search |
| Frontend DSL | `sgl.function` programs with `gen`, `select`, `fork`/`join`, control flow | Express branching/parallel multi-call generation concisely |
| Constrained decoding | JSON-schema / regex / EBNF grammar via compressed FSM + `xgrammar` | Reliable structured output with minimal throughput loss |
| Continuous batching | Token-level scheduling; requests join/leave the running batch | High GPU utilization under mixed-length traffic |
| Paged KV cache | Token-attention / paged memory management | Less fragmentation, larger effective batch |
| CUDA graphs + FlashInfer | Captured kernels and optimized attention (FlashInfer/Triton backends) | Low per-token latency, high decode throughput |
| Chunked prefill | Long prompts processed in chunks, interleaved with decode | Avoids long prefill stalls; smoother latency |
| Tensor / data parallelism | `--tp` shards a model across GPUs; `--dp` replicates for throughput | Serve larger models and scale out |
| Quantization | FP8, AWQ, GPTQ, and FP8 KV cache | Fit bigger models, faster decode |
| OpenAI-compatible server | `/v1/chat/completions`, `/v1/completions`, `/v1/embeddings` | Reuse OpenAI SDK and ecosystem tooling |
| Multimodal (VLM) support | Serves LLaVA, Qwen-VL, InternVL, and similar | One stack for text + vision

## Architecture Overview

SGLang separates a thin **frontend** (DSL programs / OpenAI clients / CLI) from the **SRT runtime**, which schedules work over the GPU engine with RadixAttention at its core.

```
        Clients / Frontend                    SRT runtime (server process)              GPU engine
 ┌──────────────────────────┐        ┌───────────────────────────────────────┐   ┌────────────────────┐
 │ OpenAI SDK / curl        │──HTTP─▶│ Tokenizer ─▶ Scheduler (continuous     │   │ Attention kernels   │
 │ sgl.function DSL program │──RPC──▶│   batching, chunked prefill)           │──▶│  (FlashInfer/Triton)│
 │ sglang.launch_server CLI │──────▶ │ RadixAttention KV cache (radix tree)   │   │ CUDA graphs         │
 └──────────────────────────┘        │ Constrained-decoding FSM (xgrammar)    │   │ Paged KV memory     │
                                      └───────────────────────────────────────┘   └────────────────────┘
                                              │ TP / DP across GPUs
                                              ▼
                                      multiple GPU workers
```

### Components

1. **Frontend** — three ways in: the OpenAI-compatible HTTP API, the `sgl.function` Python DSL (executed by a backend that points at a runtime or a remote endpoint), and the `python -m sglang.launch_server` CLI.
2. **Scheduler** — performs continuous (token-level) batching and chunked prefill, deciding which requests run each step.
3. **RadixAttention KV cache** — a radix tree keyed on token sequences; new requests that share a prefix with cached entries reuse those KV blocks instead of recomputing them. Eviction is LRU over the tree.
4. **Constrained-decoding engine** — compiles JSON schema / regex / EBNF into a (compressed) finite-state machine (via `xgrammar`) that masks logits so only valid tokens are sampled.
5. **GPU engine** — paged KV memory, FlashInfer or Triton attention kernels, and CUDA-graph-captured decode steps, optionally sharded with tensor parallelism (`--tp`) and replicated with data parallelism (`--dp`).

## Installation

### Prerequisites

- **Linux** with an NVIDIA GPU (CUDA 12.x recommended; AMD ROCm and some CPU paths exist but are less common). Compute capability 7.5+ (Turing) or newer for the fast kernels.
- **Python 3.9–3.12**.
- A matching **PyTorch** build; SGLang pulls in `flashinfer` for its fastest attention backend.
- ~16 GB+ VRAM for a 7B/8B model in FP16; far less with FP8/AWQ.

### Installation Steps

SGLang ships wheels on PyPI. The `[all]` extra installs the runtime plus FlashInfer and serving dependencies.

**Note**: Uncomment the cell below to install (e.g. in Google Colab with a GPU runtime).

In [ ]:
# Uncomment to install (Linux + NVIDIA GPU):
# !pip install "sglang[all]"

# FlashInfer (fastest attention backend) — install the wheel matching your CUDA/torch:
# !pip install flashinfer-python -i https://flashinfer.ai/whl/cu124/torch2.5/

# Verify the install and the server CLI:
# !python -c "import sglang; print(sglang.__version__)"
# !python -m sglang.launch_server --help

## Basic Usage

### Quick Start Example

The most common entry point is the **OpenAI-compatible server**. Launch it from the shell, then call it with any OpenAI client:

```bash
# Launch SRT as an OpenAI-compatible server on :30000
python -m sglang.launch_server \
    --model-path meta-llama/Llama-3.1-8B-Instruct \
    --host 0.0.0.0 --port 30000 \
    --tp 1 \
    --mem-fraction-static 0.85
```

The server exposes `/v1/chat/completions`, `/v1/completions`, `/v1/embeddings`, plus `/health` and `/get_model_info`. The cell below calls it with the OpenAI SDK.

In [ ]:
# Call the SGLang server with the OpenAI Python SDK.
# Assumes `python -m sglang.launch_server --model-path ... --port 30000` is running.
from openai import OpenAI

client = OpenAI(api_key="EMPTY", base_url="http://localhost:30000/v1")

# The model id is the served name; query it if unsure:
model_id = client.models.list().data[0].id

resp = client.chat.completions.create(
    model=model_id,
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "In one sentence, what is RadixAttention?"},
    ],
    temperature=0.3,
    max_tokens=128,
)
print(resp.choices[0].message.content)

You can also drive SGLang **without a separate server** using the offline `Engine` API — useful for batch generation and notebooks:

```python
import sglang as sgl

llm = sgl.Engine(model_path="meta-llama/Llama-3.1-8B-Instruct")
out = llm.generate(
    ["Explain continuous batching in one sentence.",
     "Write a haiku about GPU memory."],
    {"temperature": 0.7, "max_new_tokens": 128},
)
for o in out:
    print(o["text"]); print("-" * 40)
llm.shutdown()
```

In [ ]:
# The SGLang frontend DSL: a multi-step generation program.
# `s += sgl.gen(...)` calls the backend; `sgl.select` constrains to a choice set.
import sglang as sgl

@sgl.function
def triage(s, ticket):
    s += "You are a support router.\n"
    s += "Ticket: " + ticket + "\n"
    s += "Category: " + sgl.gen("category",
                                choices=["billing", "bug", "feature", "other"])
    s += "\nReply: " + sgl.gen("reply", max_tokens=80, stop="\n")

# Point the DSL at a running server (or a local Engine backend):
sgl.set_default_backend(sgl.RuntimeEndpoint("http://localhost:30000"))

state = triage.run(ticket="The app crashes when I upload a 2GB file.")
print("category:", state["category"])
print("reply:", state["reply"])

## Advanced Features

### Constrained decoding, parallelism, and prefix reuse

#### Structured / constrained output

SGLang can force output to match a **JSON schema**, **regex**, or **EBNF grammar**. With the OpenAI API this is exposed through `response_format` (JSON schema); the native API also accepts a `regex` or `ebnf` sampling parameter. Constraints are compiled to a finite-state machine (via `xgrammar`) that masks invalid tokens, so structured output stays fast.

#### Parallelism in the frontend

Inside a `sgl.function`, `s.fork(n)` launches `n` parallel branches that share the prompt prefix (so RadixAttention reuses its KV cache) and can be joined back — ideal for self-consistency, multi-sample voting, or map-style fan-out.

#### Automatic prefix caching

RadixAttention is on by default: identical leading tokens across requests (system prompt, few-shot block, conversation history) hit the cache automatically. You can disable it with `--disable-radix-cache` to measure its effect.

In [ ]:
# (a) JSON-schema-constrained output via the OpenAI-compatible API.
from openai import OpenAI
client = OpenAI(api_key="EMPTY", base_url="http://localhost:30000/v1")
model_id = client.models.list().data[0].id

schema = {
    "type": "object",
    "properties": {
        "title": {"type": "string"},
        "priority": {"type": "string", "enum": ["low", "medium", "high"]},
    },
    "required": ["title", "priority"],
    "additionalProperties": False,
}
resp = client.chat.completions.create(
    model=model_id,
    messages=[{"role": "user", "content": "Summarize: server is down for all users."}],
    response_format={"type": "json_schema",
                     "json_schema": {"name": "ticket", "schema": schema}},
    max_tokens=128,
)
print(resp.choices[0].message.content)  # guaranteed to parse as the schema


# (b) Parallel branches in the frontend DSL share the prefix via RadixAttention.
import sglang as sgl

@sgl.function
def self_consistency(s, question, n=5):
    s += "Q: " + question + "\nThink step by step, then answer.\n"
    forks = s.fork(n)                       # n branches share the cached prefix
    for f in forks:
        f += "Reasoning: " + sgl.gen("steps", max_tokens=128, temperature=0.8)
        f += "\nFinal: " + sgl.gen("ans", max_tokens=8)
    return [f["ans"] for f in forks]

## Use Cases

### Real-world Applications of SGLang

#### Use Case 1: Shared-prompt serving (RAG, agents, few-shot)

- **Context**: Every request prepends the same large system prompt, few-shot exemplars, or retrieved context.
- **Implementation**: Serve with default RadixAttention; keep the shared block at the *front* of the prompt so prefixes match.
- **Results**: The shared prefix is encoded once and reused, cutting prefill cost and raising throughput substantially.

#### Use Case 2: Reliable structured extraction at scale

- **Context**: An extraction or function-calling service must always return valid JSON conforming to a schema.
- **Implementation**: Use `response_format` JSON schema (or native `regex`/`ebnf`) so the FSM masks invalid tokens.
- **Results**: 100% parseable output with negligible throughput penalty — no retry-on-parse-failure loops.

#### Use Case 3: Agentic / multi-step generation programs

- **Context**: A workflow makes several dependent or parallel LLM calls (plan → branch → vote → synthesize).
- **Implementation**: Express it as a `sgl.function` with `gen`, `select`, and `fork`/`join`; the runtime co-optimizes caching across calls.
- **Results**: Concise program, with branches sharing KV cache and the scheduler batching across them.

## Best Practices

### Recommended Practices for SGLang

1. **Put shared content at the front of the prompt.** RadixAttention reuses *leading* tokens. Keep the system prompt, few-shot block, and retrieved context as a stable prefix so cache hits are maximized; push per-request, varying text to the end.
2. **Tune `--mem-fraction-static` deliberately.** It's the fraction of GPU memory reserved for the static model + KV pool (default ~0.85–0.9). Raise it for bigger batches/contexts; lower it on GPUs shared with other processes to avoid OOM.
3. **Use constrained decoding instead of parse-and-retry.** JSON-schema/regex/EBNF constraints guarantee valid output in one pass — cheaper and more reliable than generating freely and re-prompting on failure.
4. **Pick the right attention backend.** FlashInfer is fastest on supported GPUs; fall back to Triton (`--attention-backend triton`) if FlashInfer isn't available for your CUDA/arch.
5. **Quantize for memory-bound serving.** FP8 (weights and/or KV cache) or AWQ/GPTQ lets larger models fit and speeds decode; validate quality on your task first.
6. **Scale with `--tp` for size, `--dp` for throughput.** Tensor parallelism shards one model across GPUs to fit it; data parallelism replicates the model to serve more concurrent requests.

## Common Pitfalls

### What to Avoid When Using SGLang

1. **Variable text before the shared prefix.** Prepending a per-request id, timestamp, or user name *ahead* of the system prompt breaks prefix matching and silently kills RadixAttention hits. Keep the stable block first.
2. **Over-reserving GPU memory.** `--mem-fraction-static` near 1.0 on a shared GPU causes OOM once concurrency rises. Leave headroom for other processes and activation memory.
3. **Expecting constrained decoding to fix prompts.** A schema/regex guarantees *shape*, not *correctness*. If the prompt is ambiguous, you'll get well-formed but wrong values.
4. **Invalid tensor-parallel size.** `--tp N` must divide the model's attention-head count and the GPU count; a mismatch fails at startup.
5. **Mismatched FlashInfer / CUDA / torch builds.** The fast kernels are tied to specific CUDA and PyTorch versions; an incompatible wheel falls back to slower paths or errors. Pin them together.

## Performance Optimization

### Optimizing SGLang for Production

#### Configuration Tuning

Key parameters to optimize:

- **`--mem-fraction-static`** — fraction of GPU memory for the model + KV cache pool; the biggest lever on max batch size and prefix-cache capacity.
- **`--max-running-requests`** — caps concurrent sequences in the batch; raise to push throughput until you hit the KV or latency ceiling.
- **`--chunked-prefill-size`** — chunk length for long-prompt prefill; smaller chunks interleave decode more smoothly (lower TTFT variance), larger chunks maximize prefill throughput.
- **`--tp` / `--dp`** — tensor parallelism to fit larger models; data parallelism to scale aggregate throughput.
- **`--attention-backend`** — `flashinfer` (default, fastest) vs `triton` (broad compatibility).
- **`--enable-torch-compile`** — extra kernel fusion for small/medium models.
- **`--disable-radix-cache`** — turn off prefix caching to A/B its benefit on your workload.

In [ ]:
# Micro-benchmark: throughput with vs without RadixAttention prefix reuse.
# Run SGLang's official benchmark against a live server:
#
#   python -m sglang.bench_serving \
#       --backend sglang --host 127.0.0.1 --port 30000 \
#       --dataset-name random --num-prompts 1000 \
#       --random-input-len 1024 --random-output-len 256
#
# Quick client-side timing of shared-prefix reuse:
import time
from openai import OpenAI

client = OpenAI(api_key="EMPTY", base_url="http://localhost:30000/v1")
model_id = client.models.list().data[0].id

SYSTEM = "You are an expert assistant. " * 200  # large shared prefix
questions = [f"State fact #{i} about GPUs in one sentence." for i in range(32)]

t0 = time.perf_counter()
for q in questions:
    client.chat.completions.create(
        model=model_id,
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": q}],
        max_tokens=48, temperature=0.0,
    )
dt = time.perf_counter() - t0
print(f"{len(questions)} requests sharing a {len(SYSTEM)}-char prefix in {dt:.2f}s")
print("After the first request the prefix is cached -> later prefills are near-free.")

## Production Deployment

### Deploying SGLang in Production

#### Docker Deployment

SGLang publishes official CUDA images (`lmsysorg/sglang`). Mount your model cache and expose the server port; `--ipc=host` is recommended for shared-memory.

```dockerfile
FROM lmsysorg/sglang:latest

ENV HF_HOME=/root/.cache/huggingface
EXPOSE 30000

ENTRYPOINT ["python", "-m", "sglang.launch_server", \
            "--model-path", "meta-llama/Llama-3.1-8B-Instruct", \
            "--host", "0.0.0.0", "--port", "30000", \
            "--mem-fraction-static", "0.85"]
```

```bash
docker run --gpus all --ipc=host \
    -v ~/.cache/huggingface:/root/.cache/huggingface \
    -p 30000:30000 \
    lmsysorg/sglang:latest \
    python -m sglang.launch_server \
        --model-path meta-llama/Llama-3.1-8B-Instruct \
        --host 0.0.0.0 --port 30000
```

#### Kubernetes Deployment

Request a GPU via `nvidia.com/gpu`, mount a shared-memory volume, and probe the `/health` endpoint.

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: sglang-llama
spec:
  replicas: 1
  selector:
    matchLabels: { app: sglang-llama }
  template:
    metadata:
      labels: { app: sglang-llama }
    spec:
      containers:
        - name: sglang
          image: lmsysorg/sglang:latest
          args:
            - python
            - -m
            - sglang.launch_server
            - --model-path=meta-llama/Llama-3.1-8B-Instruct
            - --host=0.0.0.0
            - --port=30000
            - --mem-fraction-static=0.85
          ports:
            - containerPort: 30000
          resources:
            limits:
              nvidia.com/gpu: 1
          volumeMounts:
            - { name: dshm, mountPath: /dev/shm }
          readinessProbe:
            httpGet: { path: /health, port: 30000 }
            initialDelaySeconds: 60
            periodSeconds: 10
      volumes:
        - name: dshm
          emptyDir: { medium: Memory, sizeLimit: 8Gi }
---
apiVersion: v1
kind: Service
metadata:
  name: sglang-llama
spec:
  selector: { app: sglang-llama }
  ports:
    - port: 80
      targetPort: 30000
```

## Monitoring and Observability

### Monitoring SGLang in Production

#### Key Metrics to Track

- **Throughput (tokens/sec & requests/sec)** — headline capacity; SGLang logs running/queued request counts and token rates.
- **Latency** — time-to-first-token (TTFT) and inter-token latency (ITL); TTFT is dominated by prefill and benefits most from prefix caching.
- **Cache hit rate** — the fraction of prefill tokens served from the RadixAttention cache; a low rate on a supposedly prefix-heavy workload signals prefix mismatch.
- **KV-cache / GPU memory utilization** — sustained saturation means you're batch-limited; consider quantization, more GPUs, or a higher `--mem-fraction-static`.
- **Queue depth / running requests** — rising queues indicate overload; scale out with `--dp` or more replicas.

#### Logging & Metrics Best Practices

- Launch with `--enable-metrics` to expose a **Prometheus** `/metrics` endpoint (throughput, latency histograms, cache stats); scrape into Grafana.
- Pair with NVIDIA **DCGM-Exporter** for GPU utilization, memory, and temperature.
- Set log verbosity with `--log-level` (e.g. `info`/`debug`) and ship stdout/stderr to your log pipeline.
- Front the OpenAI-compatible endpoint with a gateway (NGINX/Envoy) that records status codes and latency percentiles.

## Troubleshooting

### Common Issues with SGLang

#### Issue 1: CUDA out-of-memory at startup or under load

**Symptoms**: `CUDA out of memory` when loading the model or as concurrency grows.

**Cause**: `--mem-fraction-static` reserves too much (or too little leaving no room for activations), the model is too large for the GPU, or another process shares the card.

**Solution**: Lower `--mem-fraction-static`, reduce `--max-running-requests`, enable FP8/AWQ weight or FP8 KV-cache quantization, or shard with a larger `--tp`.

#### Issue 2: RadixAttention isn't speeding anything up

**Symptoms**: Throughput on a prefix-heavy workload is no better than expected; cache hit rate is low.

**Cause**: Per-request variable text appears *before* the shared block, so prefixes don't match; or the prefix is shorter than expected.

**Solution**: Reorder prompts so the stable shared content leads; verify with `--disable-radix-cache` (A/B) and by watching the cache hit rate in the logs/metrics.

#### Issue 3: FlashInfer import or kernel error on startup

**Symptoms**: An import error or kernel failure mentioning FlashInfer when the server starts.

**Cause**: The FlashInfer wheel doesn't match the installed CUDA / PyTorch version, or the GPU arch is unsupported.

**Solution**: Install the FlashInfer wheel for your exact CUDA/torch, or run with `--attention-backend triton` as a compatible fallback.

#### Issue 4: Constrained output is slow or rejected

**Symptoms**: JSON/regex-constrained requests are much slower or error on a complex schema.

**Cause**: A very large or deeply recursive grammar inflates FSM compilation; an unsupported schema feature.

**Solution**: Simplify the schema/grammar, reuse it across requests so compilation is amortized, and keep `xgrammar` updated.

## Comparison with Alternatives

### How SGLang Compares to Other Solutions

| Feature | SGLang | vLLM | TGI (Text Generation Inference) |
|---------|--------|------|---------------------------------|
| Core engine | SRT (Python + CUDA kernels, FlashInfer/Triton) | Python + custom CUDA (PagedAttention) | Rust server + PyTorch/CUDA |
| Signature optimization | RadixAttention (automatic prefix KV reuse) | PagedAttention; prefix caching opt-in | Prefix caching, flash attention |
| Continuous batching | Yes | Yes | Yes |
| Structured decoding | JSON / regex / EBNF via compressed FSM (xgrammar) | JSON/regex (outlines/xgrammar backends) | JSON/regex (grammar) |
| Frontend DSL | Yes — `sgl.function` programs with fork/join | No (API only) | No (API only) |
| Quantization | FP8, AWQ, GPTQ, FP8 KV cache | AWQ, GPTQ, FP8, others | AWQ, GPTQ, EETQ, others |
| OpenAI-compatible API | Yes | Yes | Yes (and TGI-native) |
| Standout strength | Automatic prefix reuse + programmable multi-call generation | Huge model/feature coverage, large community | Tight HF Hub integration, production hardening |

### When to Choose This Tool

Choose SGLang when:

- Your workload is **prefix-heavy** (shared system prompts, few-shot, RAG, multi-turn, tree search) and RadixAttention's automatic reuse pays off.
- You need **fast, reliable structured output** (JSON/regex/grammar) at serving scale.
- You're writing **multi-step or agentic generation programs** and want a DSL the runtime co-optimizes.
- You want **top-tier throughput** with an OpenAI-compatible endpoint and minimal setup.

## Resources

### Official Documentation

- Official docs: <https://docs.sglang.ai>
- GitHub repository: <https://github.com/sgl-project/sglang>
- RadixAttention paper (*Efficiently Programming LLMs with SGLang*): <https://arxiv.org/abs/2312.07104>

### Tutorials and Guides

- OpenAI-compatible server & sampling parameters: <https://docs.sglang.ai/backend/openai_api_completions.html>
- Structured / constrained decoding: <https://docs.sglang.ai/backend/structured_outputs.html>
- Frontend language (DSL) guide: <https://docs.sglang.ai/frontend/frontend.html>

### Community Resources

- GitHub Issues & Discussions: <https://github.com/sgl-project/sglang/issues>
- SGLang project (LMSYS) org: <https://github.com/sgl-project>
- PyPI package: <https://pypi.org/project/sglang/>

### Related Technologies

- **RadixAttention** — the automatic prefix KV-cache reuse at the core of SRT.
- **xgrammar** — the fast grammar/FSM backend powering constrained decoding.
- **FlashInfer** — the optimized attention kernel library SGLang uses by default.
- **vLLM, TGI, LMDeploy, TensorRT-LLM** — alternative high-throughput LLM serving stacks.